# 02 · Quy tắc re-rank top-5 trên SMLP4Rec (cluster đã đặt vs cluster mới)

Các quy tắc hậu truy xuất áp dụng lên trọng số đã huấn luyện ở notebook 01. Thay thế `scripts/run_smlprec_expedia_mixed.py` (branch `exp/fixed-quota-2past-3novel`) và `scripts/run_smlprec_expedia_dynamic_cap.py` (branch `exp/dynamic-past-cap-by-history-length`).

Các quy tắc không thay đổi việc huấn luyện, nên các script đã huấn luyện lại cùng seed từ đầu (và kiểm tra trọng số khớp với lượt chạy plain). Ở đây nạp trọng số từng epoch do notebook 01 lưu, cho cùng trọng số mà không cần huấn luyện lại.

- Cluster **past** = cluster có trong lịch sử của user (đầu vào mô hình, gồm booking gần nhất); **novel** = không có trong lịch sử. **L** = độ dài lịch sử (số booking trước đó, giới hạn 20).
- **Fixed quota**: top 5 = `min(2, số cluster past khác nhau)` cluster past điểm cao nhất + các cluster novel điểm cao nhất cho các vị trí còn lại, xếp theo điểm.
- **Dynamic cap**: top 5 = 5 cluster điểm cao nhất với tối đa 1 cluster past nếu L < 5 và tối đa 2 nếu L >= 5 (cluster past vẫn cạnh tranh theo điểm).
- **Metric**: chỉ Recall@5 và NDCG@5 (các quy tắc định nghĩa danh sách 5 item), tổng thể, trên các dòng có cluster tiếp theo thật là cũ (có trong lịch sử) hoặc mới, và theo lát cắt L.

Kernel: `smlp4rec`. Cần thư mục chạy của notebook 01.

In [1]:
import functools
import json
import logging
import sys
from pathlib import Path

import numpy as np
import torch  # before pandas (Windows c10.dll issue)

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
torch.load = functools.partial(torch.load, weights_only=False)

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed

from nbp.eval.compare import compare_nested
from nbp.eval.metrics import rank_in_list, summarize
from src.models.smlprec import SMLPREC

torch.autograd.set_detect_anomaly(False)
logging.disable(logging.INFO)

CONFIG = ROOT / "configs" / "smlprec_expedia.yaml"
WORK = ROOT / "data" / "interim" / "recbole"
REF_DIR = ROOT / "results" / "week3_implementation"
# latest notebook-01 run; set explicitly to pin a run
RUN_DIR = sorted((ROOT / "experiments").glob("*_expedia_smlp4rec_clean-plain"))[-1]
K = 5
print("run dir:", RUN_DIR)

run dir: C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-08_expedia_smlp4rec_clean-plain


## 1. Dữ liệu (cùng config và cách chia với notebook 01)

In [2]:
config = Config(
    model=SMLPREC,
    dataset="expedia_clean",
    config_file_list=[str(CONFIG)],
    config_dict={"data_path": str(WORK), "checkpoint_dir": str(RUN_DIR)},
)
init_seed(config["seed"], config["reproducibility"])
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
n_items = dataset.item_num
ITEM_SEQ = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
ITEM_SEQ_LEN = config["ITEM_LIST_LENGTH_FIELD"]


def split_inputs(loader) -> dict:
    """History, L, target, and in_hist (n, n_items) bool: cluster appears in the history."""
    d = loader.dataset.inter_feat
    seq, length = d[ITEM_SEQ], d[ITEM_SEQ_LEN]
    y = d[config["ITEM_ID_FIELD"]].numpy()
    seq_np, len_np = seq.numpy(), length.numpy()
    in_hist = np.zeros((len(y), n_items), bool)
    for j in range(seq_np.shape[1]):
        r = np.flatnonzero(j < len_np)
        in_hist[r, seq_np[r, j]] = True
    in_hist[:, 0] = False  # padding id
    old = in_hist[np.arange(len(y)), y]
    return {"seq": seq, "length": length, "y": y, "in_hist": in_hist, "old": old}


inputs = {"valid": split_inputs(valid_data), "test": split_inputs(test_data)}
for sp, a in inputs.items():
    print(f"{sp}: {len(a['y'])} rows, old-target share {a['old'].mean():.4f}")

valid: 213869 rows, old-target share 0.2970
test: 213869 rows, old-target share 0.2919


## 2. Các quy tắc

Cả hai nhận ma trận điểm đầy đủ (cột padding = -inf) và mặt nạ cluster past, trả về danh sách (n, 5) xếp theo điểm.

In [3]:
N_PAST_MAX = 2
L_THRESHOLD, CAP_LOW, CAP_HIGH = 5, 1, 2
NEG = torch.tensor(float("-inf"))


def plain_top5(scores: torch.Tensor, in_hist: torch.Tensor, length: torch.Tensor) -> torch.Tensor:
    """The 5 highest scores."""
    return torch.topk(scores, K, dim=1).indices


def fixed_quota_top5(scores, in_hist, length) -> torch.Tensor:
    """n_past = min(2, distinct past clusters) best past + (5 - n_past) best novel, ordered by score."""
    past_v, past_i = torch.topk(torch.where(in_hist, scores, NEG), N_PAST_MAX, dim=1)
    novel_v, novel_i = torch.topk(torch.where(in_hist, NEG, scores), K - 1, dim=1)
    n_past = in_hist.sum(1).clamp(max=N_PAST_MAX)  # >= 1 on every target row
    take_past = torch.arange(N_PAST_MAX)[None, :] < n_past[:, None]
    take_novel = torch.arange(K - 1)[None, :] < (K - n_past)[:, None]
    cand_v = torch.cat(
        [past_v.masked_fill(~take_past, float("-inf")), novel_v.masked_fill(~take_novel, float("-inf"))], 1
    )
    cand_i = torch.cat([past_i, novel_i], 1)
    return torch.gather(cand_i, 1, torch.topk(cand_v, K, dim=1).indices)


def dynamic_cap_top5(scores, in_hist, length) -> torch.Tensor:
    """5 best clusters with at most 1 past cluster if L < 5, at most 2 if L >= 5."""
    past_v, past_i = torch.topk(torch.where(in_hist, scores, NEG), N_PAST_MAX, dim=1)
    novel_v, novel_i = torch.topk(torch.where(in_hist, NEG, scores), K, dim=1)
    cap = torch.where(length < L_THRESHOLD, CAP_LOW, CAP_HIGH)
    allow_past = torch.arange(N_PAST_MAX)[None, :] < cap[:, None]
    cand_v = torch.cat([past_v.masked_fill(~allow_past, float("-inf")), novel_v], 1)
    cand_i = torch.cat([past_i, novel_i], 1)
    return torch.gather(cand_i, 1, torch.topk(cand_v, K, dim=1).indices)


RULES = {"plain": plain_top5, "mixed": fixed_quota_top5, "capped": dynamic_cap_top5}

## 3. Chấm điểm và metric

In [4]:
@torch.no_grad()
def top5_lists(model, a: dict, batch: int = 8192) -> dict[str, np.ndarray]:
    """(n, 5) list per rule for every row of one split, current weights."""
    model.eval()
    out = {name: [] for name in RULES}
    for i in range(0, len(a["y"]), batch):
        s = model.full_sort_predict(
            {model.ITEM_SEQ: a["seq"][i : i + batch], model.ITEM_SEQ_LEN: a["length"][i : i + batch]}
        )
        s[:, 0] = float("-inf")
        h = torch.from_numpy(a["in_hist"][i : i + batch])
        for name, rule in RULES.items():
            out[name].append(rule(s, h, a["length"][i : i + batch]).numpy())
    return {name: np.concatenate(v) for name, v in out.items()}


def rule_metrics(top5: np.ndarray, a: dict, slices: dict[str, np.ndarray]) -> dict:
    """Recall@5 / NDCG@5 overall, on old- and new-target rows, and per L slice."""
    rank = rank_in_list(top5, a["y"])
    allm = summarize(rank, ks=(K,))
    oldm, newm = summarize(rank, a["old"], ks=(K,)), summarize(rank, ~a["old"], ks=(K,))
    out = {
        "recall@5": allm["recall@5"],
        "ndcg@5": allm["ndcg@5"],
        "old_target_hit_of_old_rows": oldm["recall@5"],  # = Recall@5 on old-target rows
        "new_target_hit_of_new_rows": newm["recall@5"],  # = Recall@5 on new-target rows
        "ndcg@5_old_rows": oldm["ndcg@5"],
        "ndcg@5_new_rows": newm["ndcg@5"],
    }
    for name, m in slices.items():
        sm = summarize(rank, m, ks=(K,))
        out[f"recall@5_{name}"], out[f"ndcg@5_{name}"] = sm["recall@5"], sm["ndcg@5"]
    return out


def slices_of(a: dict) -> dict[str, np.ndarray]:
    L = a["length"].numpy()
    return {"history_len_1": L == 1, "history_len_2plus": L >= 2, "L_lt_5": L < L_THRESHOLD, "L_ge_5": L >= L_THRESHOLD}

## 4. Đánh giá trọng số của mọi epoch từ notebook 01

Epoch tốt nhất theo từng quy tắc = NDCG@5 **valid** cao nhất dưới quy tắc đó (như trong các script); test được đọc ở epoch đó.

In [5]:
model = SMLPREC(config, train_data.dataset)
n_epochs = config["epochs"]
per_epoch = []
for epoch in range(1, n_epochs + 1):
    model.load_state_dict(torch.load(RUN_DIR / f"epoch{epoch}.pth"))
    row = {"epoch": epoch}
    for sp, a in inputs.items():
        lists = top5_lists(model, a)
        row[sp] = {name: rule_metrics(lists[name], a, slices_of(a)) for name in RULES}
    per_epoch.append(row)
    print(
        f"epoch {epoch} test R@5 / NDCG@5: "
        + " | ".join(f"{n} {row['test'][n]['recall@5']:.4f} / {row['test'][n]['ndcg@5']:.4f}" for n in RULES)
    )

best = {name: max(per_epoch, key=lambda r: r["valid"][name]["ndcg@5"])["epoch"] for name in RULES}
print("best epoch by valid NDCG@5:", best)

epoch 1 test R@5 / NDCG@5: plain 0.3214 / 0.2274 | mixed 0.2991 / 0.2174 | capped 0.2893 / 0.2121


epoch 2 test R@5 / NDCG@5: plain 0.3212 / 0.2274 | mixed 0.2986 / 0.2173 | capped 0.2886 / 0.2119


epoch 3 test R@5 / NDCG@5: plain 0.3212 / 0.2274 | mixed 0.2985 / 0.2172 | capped 0.2887 / 0.2119
best epoch by valid NDCG@5: {'plain': 3, 'mixed': 3, 'capped': 3}


## 5. Bảng kết quả (test, epoch tốt nhất theo từng quy tắc)

In [6]:
import pandas as pd

table = pd.DataFrame(
    {name: per_epoch[best[name] - 1]["test"][name] for name in RULES}
).T
table.insert(0, "epoch", [best[n] for n in RULES])
table

,epoch,recall@5,ndcg@5,old_target_hit_of_old_rows,new_target_hit_of_new_rows,ndcg@5_old_rows,ndcg@5_new_rows,recall@5_history_len_1,ndcg@5_history_len_1,recall@5_history_len_2plus,ndcg@5_history_len_2plus,recall@5_L_lt_5,ndcg@5_L_lt_5,recall@5_L_ge_5,ndcg@5_L_ge_5
plain,3,0.3212,0.2274,0.7995,0.1240,0.6335,0.0600,0.2528,0.1798,0.3368,0.2383,0.2827,0.1997,0.3597,0.2552
mixed,3,0.2985,0.2172,0.6377,0.1587,0.5602,0.0759,0.2528,0.1798,0.3089,0.2258,0.2796,0.1982,0.3174,0.2363
capped,3,0.2887,0.2119,0.5740,0.1710,0.5230,0.0836,0.2528,0.1798,0.2969,0.2192,0.2601,0.1876,0.3173,0.2362


## 6. Lưu và so sánh với các lượt chạy của script

JSON của script chứa, theo mỗi epoch, `plain` cộng một quy tắc (`mixed` hoặc `capped`). Kết quả của notebook được định dạng lại theo cùng bố cục và mọi giá trị số chung đều được so sánh (MRR và số slot của script không được tính lại ở đây).

In [7]:
def as_script_layout(rule: str) -> dict:
    return {
        "per_epoch": [
            {"epoch": r["epoch"], **{sp: {"plain": r[sp]["plain"], rule: r[sp][rule]} for sp in inputs}}
            for r in per_epoch
        ],
        f"best_epoch_by_valid_{rule}_ndcg@5": best[rule],
        "best_epoch_by_valid_plain_ndcg@5": best["plain"],
        f"final_test_at_best_{rule}_epoch": per_epoch[best[rule] - 1]["test"],
    }


result = {"run_dir": str(RUN_DIR), "rules": list(RULES), "best_epoch_by_valid_ndcg@5": best, "per_epoch": per_epoch}
(RUN_DIR / "top5_rules.json").write_text(json.dumps(result, indent=2), encoding="utf-8")

rows = []
for rule, ref_file in (("mixed", "smlprec_expedia_mixed_run.json"), ("capped", "smlprec_expedia_dynamic_cap_run.json")):
    ref = json.loads((REF_DIR / ref_file).read_text(encoding="utf-8"))
    rows += [(ref_file, *r) for r in compare_nested(ref, as_script_layout(rule))]
cmp = pd.DataFrame(rows, columns=["reference", "path", "script", "notebook", "diff"])
pd.set_option("display.max_rows", 300)
# INFORMATIONAL: the reference JSONs belong to the uncollapsed data, differences are expected.
print(f"compared {len(cmp)} values; identical: {(cmp['diff'].abs() < 1e-9).sum()}; max |diff|: {cmp['diff'].abs().max():.6f}")
cmp[cmp["diff"].abs() >= 1e-9] if (cmp["diff"].abs() >= 1e-9).any() else cmp.head(20)

compared 178 values; identical: 12; max |diff|: 0.030600


,reference,path,script,notebook,diff
1,smlprec_expedia_mixed_run.json,per_epoch.0.valid.plain.recall@5,0.3325,0.3203,-0.0122
2,smlprec_expedia_mixed_run.json,per_epoch.0.valid.plain.ndcg@5,0.2409,0.2274,-0.0135
3,smlprec_expedia_mixed_run.json,per_epoch.0.valid.plain.old_target_hit_of_old_...,0.8100,0.7915,-0.0185
4,smlprec_expedia_mixed_run.json,per_epoch.0.valid.plain.new_target_hit_of_new_...,0.1166,0.1213,0.0047
5,smlprec_expedia_mixed_run.json,per_epoch.0.valid.plain.recall@5_history_len_1,0.2569,0.2452,-0.0117
6,smlprec_expedia_mixed_run.json,per_epoch.0.valid.plain.recall@5_history_len_2...,0.3489,0.3369,-0.0120
7,smlprec_expedia_mixed_run.json,per_epoch.0.valid.mixed.recall@5,0.3077,0.2968,-0.0109
8,smlprec_expedia_mixed_run.json,per_epoch.0.valid.mixed.ndcg@5,0.2298,0.2169,-0.0129
9,smlprec_expedia_mixed_run.json,per_epoch.0.valid.mixed.old_target_hit_of_old_...,0.6524,0.6324,-0.0200
10,smlprec_expedia_mixed_run.json,per_epoch.0.valid.mixed.new_target_hit_of_new_...,0.1518,0.1551,0.0033
